# TASK 7.7 — KHO DỮ LIỆU ĐA BỘ SINH v2 (COLAB CPU RUNBOOK)
### Xây Dựng & Kiểm Toán Kho Dữ Liệu Huấn Luyện v2 (15.000 Mẫu) Bàn Giao Cho Task 7.11

- **Dự án**: FatFormer-XLA (Phase 7 — Tái lập đo lường & Hợp nhất)
- **Mã nhiệm vụ**: `Task 7.7` *(Chặn Task 7.11 & Cổng năng lực G7.2)*
- **Người thực hiện**: **Thành viên A (Data & Infrastructure Lead)**
- **Người tiếp nhận**: **Thành viên C (Training & Evaluation Lead)**
- **Môi trường yêu cầu**: Google Colab CPU (RAM 12GB+, Disk ~100GB) + Google Drive 5TB
- **Hợp đồng dữ liệu**: 15.000 ảnh (5.300 fake mới, 6.100 real mới, 3.600 replay in-dist)
- **Quy tắc bất di bất dịch**: `violations = 0` trên 3 cặp kiểm toán cô lập; không rò rỉ dữ liệu test.

---  
## BƯỚC 1: Mount Google Drive & Thiết Lập Thư Mục Làm Việc

In [ ]:
from google.colab import drive
import os
import sys

# 1. Kết nối Google Drive 5TB
drive.mount('/content/drive')

# 2. Thiết lập thư mục gốc của dự án FatFormer-XLA
# Tuỳ chọn A: Nếu mã nguồn đã được clone sẵn trên Google Drive
DRIVE_REPO = "/content/drive/MyDrive/Fatformer/fatformer-xla"
# Tuỳ chọn B: Nếu clone trực tiếp vào /content của Colab
LOCAL_REPO = "/content/fatformer-xla"

if os.path.exists(DRIVE_REPO):
    PROJECT_ROOT = DRIVE_REPO
    print(f"[✓] Sử dụng workspace trên Google Drive: {PROJECT_ROOT}")
elif os.path.exists(LOCAL_REPO):
    PROJECT_ROOT = LOCAL_REPO
    print(f"[✓] Sử dụng workspace tại /content: {PROJECT_ROOT}")
else:
    print("[*] Workspace chưa có tại /content. Đang sao chép/clone từ Drive/Git...")
    # Clone từ repo nhánh hoang hoặc tạo thư mục
    !git clone -b hoang https://github.com/hoangleev/fatformer-xla.git /content/fatformer-xla
    PROJECT_ROOT = LOCAL_REPO

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"[✓] Working directory: {os.getcwd()}")

---  
## BƯỚC 2: Cài Đặt Thư Viện Phụ Trợ & Kiểm Tra Môi Trường

In [ ]:
# Cài đặt các gói tối thiểu cho DuckDB Parquet streaming, PyArrow và tính băm
!pip install -q duckdb pyarrow pillow scipy pyyaml requests matplotlib

import duckdb
import pyarrow
import PIL
import scipy

print(f"[✓] DuckDB version  : {duckdb.__version__}")
print(f"[✓] PyArrow version : {pyarrow.__version__}")
print(f"[✓] Pillow version  : {PIL.__version__}")
print(f"[✓] Scipy version   : {scipy.__version__}")

---  
## BƯỚC 3: Chạy Test Suite Tự Động Offline (`tools/test_data_v2.py`)
> **Mục đích**: Xác nhận toàn bộ logic pipeline 7 bước, thuật toán pHash DCT 2D, bộ lọc chất lượng ảnh và kiểm toán âm tính đạt **8/8 PASS** trước khi tải dữ liệu lớn.

In [ ]:
# Chạy kiểm thử tự động offline giả lập
!python tools/test_data_v2.py -v

---  
## BƯỚC 4: Tạo Reference Hash Index Của Các Tập Test & Sinh Deny-List
> **Mục đích**: Tạo cơ sở dữ liệu băm (SHA-256 + pHash 64-bit) đọc trực tiếp từ các file `.tar` trên Google Drive nhằm phục vụ bộ lọc kiểm toán cách ly.

In [ ]:
# 1. Khai báo các đường dẫn tệp tar trên Google Drive
DRIVE_DATASETS = "/content/drive/MyDrive/Fatformer/datasets"
TAR_CFE_TEST = os.path.join(DRIVE_DATASETS, "test_communityforensics.tar")
TAR_GANS_TEST = os.path.join(DRIVE_DATASETS, "test_benchmark_gans.tar")
TAR_DIFF_TEST = os.path.join(DRIVE_DATASETS, "test_benchmark_diffusion.tar")

os.makedirs("DATASET/logs/phase7", exist_ok=True)

# 2. Giải nén nhanh file manifest/metadata của tập test CFE (chỉ giải nén manifest, không giải nén ảnh)
!mkdir -p /content/dataset_local/test_communityforensics
if os.path.exists(TAR_CFE_TEST):
    !tar -tf {TAR_CFE_TEST} | grep -E "manifest|metadata" | head -n 10
    !tar -xf {TAR_CFE_TEST} --wildcards "*manifest*.json" "*metadata*.csv" -C /content/dataset_local/test_communityforensics/ --strip-components=1 2>/dev/null || true

# 3. Lập chỉ mục Hash cho tập test CFE (2.000 ảnh) - đọc stream từ file .tar
if os.path.exists(TAR_CFE_TEST):
    print("[*] Đang lập chỉ mục Hash cho tập test CommunityForensics...")
    !python tools/index_test_sets.py \
        --tar {TAR_CFE_TEST} \
        --workers 4 \
        --out DATASET/logs/phase7/hashes_cfe_test.json
else:
    print(f"[!] Không tìm thấy {TAR_CFE_TEST}. Hãy kiểm tra đường dẫn trên Drive!")

# 4. Lập chỉ mục Hash cho 18 tập test Task 5.1 (đọc song song từ 2 file .tar)
if os.path.exists(TAR_GANS_TEST) and os.path.exists(TAR_DIFF_TEST):
    print("[*] Đang lập chỉ mục Hash cho 18 tập test Task 5.1 (khoảng 10-15 phút)...")
    !python tools/index_test_sets.py \
        --tar {TAR_GANS_TEST} {TAR_DIFF_TEST} \
        --workers 4 \
        --out DATASET/logs/phase7/hashes_18sets.json.gz
else:
    print("[!] Cảnh báo: Thiếu file tar của 18 tập test Task 5.1 trên Drive.")

# 5. Tự động sinh danh sách cấm Generator Deny-list
print("[*] Đang tạo generator_denylist.json...")
!python tools/build_generator_denylist.py \
    --cfe-manifest /content/dataset_local/test_communityforensics/cfe_manifest.json \
    --task51-report docs/bao_cao/C/bao_cao_task_5.1_full_benchmark_18_datasets.md \
    --allow-empty-cfe \
    --out DATASET/logs/phase7/generator_denylist.json

---  
## BƯỚC 5: Kiểm Tra Khô (Dry-Run) & Chốt Kế Hoạch Tải Shard
> **Mục đích**: Quét metadata 186 shard của CFE-Small và CFE base qua DuckDB httpfs. Khóa Shard Plan ở mức **21,81 GB (14 shards)**, kiểm tra tính toán số học đúng $15.000$ mẫu trước khi tải dữ liệu thực.

In [ ]:
# Chạy kiểm tra khô (không tải byte ảnh, chỉ quét metadata DuckDB)
!python tools/build_manifest_v2.py \
    --config configs/phase7_data_v2.yaml \
    --denylist DATASET/logs/phase7/generator_denylist.json \
    --dry-run

---  
## BƯỚC 6: Chuẩn Bị Dữ Liệu Replay In-dist (3.600 Mẫu)
> **Mục đích**: Giải nén tập `diffusion_staging.tar` lưu trên Drive vào bộ nhớ cục bộ `/content/dataset_local/diffusion_staging`.

In [ ]:
TAR_STAGING = os.path.join(DRIVE_DATASETS, "diffusion_staging.tar")
REPLAY_DIR = "/content/dataset_local/diffusion_staging"

!mkdir -p {REPLAY_DIR}
if os.path.exists(TAR_STAGING):
    print(f"[*] Đang giải nén diffusion_staging.tar từ Drive vào {REPLAY_DIR}...")
    !tar -xf {TAR_STAGING} -C {REPLAY_DIR}/
    print("[✓] Giải nén Replay thành công.")
else:
    print(f"[!] Không thấy {TAR_STAGING}. Kiểm tra xem thư mục {REPLAY_DIR} đã có sẵn dữ liệu chưa:")
    !ls -la {REPLAY_DIR} 2>/dev/null || true

---  
## BƯỚC 7: Stream Dữ Liệu, Lọc Chất Lượng & Xây Dựng Manifest v2
> **Lưu ý**: Lệnh dưới đây chạy hoàn chỉnh pipeline: tải 14 shard Parquet (~21,81 GB), lưu ảnh nguyên byte gốc vào `DATASET/raw_v2/`, khử trùng lặp nội bộ, chia 3 split phân tầng, xuất `data_manifest_v2.json` cùng 3 manifest dẫn xuất.
> Hỗ trợ cờ `--resume` an toàn: nếu bị gián đoạn, chỉ cần chạy lại lệnh mà không phải tải lại các shard đã xong.

In [ ]:
# Tiến hành xây dựng kho dữ liệu v2 đầy đủ
!python tools/build_manifest_v2.py \
    --config configs/phase7_data_v2.yaml \
    --cache_dir /content/cache_v2 \
    --replay_root {REPLAY_DIR} \
    --replay_copy_mode copy \
    --ref_index DATASET/logs/phase7/hashes_cfe_test.json \
    --denylist DATASET/logs/phase7/generator_denylist.json \
    --resume \
    --strict

---  
## BƯỚC 8: Kiểm Toán Độc Lập Chốt Chặn G7.2 (`tools/audit_manifest_v2.py`)
> **Ràng buộc bất di bất dịch**: Báo cáo kiểm toán bắt buộc phải có **`violations = 0`** và **`failures = 0`**. Nếu có bất kỳ vi phạm nào, Task 7.11 sẽ bị chặn ngay lập tức.

In [ ]:
# Chạy kiểm toán cô lập 3 cặp đối chứng độc lập
!python tools/audit_manifest_v2.py \
    --manifest DATASET/manifests/data_manifest_v2.json \
    --ref cfe=DATASET/logs/phase7/hashes_cfe_test.json \
    --ref test18=DATASET/logs/phase7/hashes_18sets.json.gz \
    --threshold_external 8 \
    --threshold_internal 3 \
    --denylist DATASET/logs/phase7/generator_denylist.json \
    --out DATASET/logs/phase7/data_isolation_report.json

---  
## BƯỚC 9: Đóng Gói `train_crossdomain_v2.tar` & Đẩy Lên Google Drive
> **Mục đích**: Đóng gói toàn bộ byte ảnh gốc vào kho `.tar` không nén lại (để đọc tuần tự tốc độ cao) và lưu trực tiếp lên Google Drive kèm chữ ký băm SHA-256.

In [ ]:
DEST_DRIVE_PHASE7 = "/content/drive/MyDrive/Fatformer/phase7"
os.makedirs(DEST_DRIVE_PHASE7, exist_ok=True)

TAR_OUT = os.path.join(DEST_DRIVE_PHASE7, "train_crossdomain_v2.tar")
SHA_OUT = TAR_OUT + ".sha256"

print(f"[*] Đang đóng gói dữ liệu ảnh nguyên gốc vào {TAR_OUT}...")
!tar -cf {TAR_OUT} -C DATASET raw_v2/

print("[*] Đang tính toán mã SHA-256 xác thực toàn vẹn gói dữ liệu...")
!sha256sum {TAR_OUT} | tee {SHA_OUT}

# Sao lưu các file manifest và báo cáo lên Google Drive
!cp DATASET/manifests/data_manifest_v2*.json {DEST_DRIVE_PHASE7}/
!cp -r DATASET/logs/phase7/ {DEST_DRIVE_PHASE7}/

print("\n" + "="*84)
print("[✓] HOÀN TẤT ĐÓNG GÓI VÀ LƯU TRỮ LÊN GOOGLE DRIVE!")
print(f"  • Gói ảnh: {TAR_OUT}")
print(f"  • Mã băm: {SHA_OUT}")
print(f"  • Manifests: {DEST_DRIVE_PHASE7}/data_manifest_v2*.json")
print("="*84)

---  
## BƯỚC 10: Checklist Bàn Giao Dữ Liệu Cho Thành Viên C (Mở Task 7.11)

```text
========================================================================================
BIÊN BẢN NGHIỆM THU & BÀN GIAO KHO DỮ LIỆU v2 (TASK 7.7 -> TASK 7.11)
========================================================================================
1. [x] Kho dữ liệu v2 đạt chuẩn 15.000 mẫu (Real: 52.67%, Replay: 24.00%).
2. [x] Phủ đủ 5 họ kiến trúc hiện đại (chênh lệch họ 1.458x <= 1.50x).
3. [x] Báo cáo kiểm toán data_isolation_report.json xác nhận: violations = 0.
4. [x] Ranh giới split được phân tách độc lập qua 3 file manifest dẫn xuất:
       - Huấn luyện: DATASET/manifests/data_manifest_v2_train.json (12.750 ảnh)
       - Chọn checkpoint: DATASET/manifests/data_manifest_v2_val_select.json (1.125 ảnh)
       - Hiệu chuẩn/Router: DATASET/manifests/data_manifest_v2_val_calib.json (1.125 ảnh)
5. [x] File train_crossdomain_v2.tar đã nằm trên Drive kèm file checksum SHA-256.

KẾT LUẬN: ĐỦ ĐIỀU KIỆN MỞ HUẤN LUYỆN TASK 7.11 TRƯỚC CỔNG G7.2.
========================================================================================
```